In [0]:
%pip install -U -qqq langgraph-supervisor==0.0.30 mlflow[databricks] databricks-langchain databricks-agents uv 
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.

Error: The "path" argument must be of type string. Received type undefined


## Define the multi-agent system

Create a multi-agent system in LangGraph using a supervisor agent node with one or more of the following subagents:
- **GenieAgent**: A LangChain runnable that allows you to easily interact with your Genie Space to query structured data.
- **Custom serving agent**: An agent that is already hosted as an existing endpoint on Databricks.
- **In-code tool-calling agent**: An agent that calls Unity Catalog function tools, defined within this notebook. This example uses `system.ai.python_exec`, but for examples of other tools you can add to your agents, see Databricks documentation ([AWS](https://docs.databricks.com/aws/generative-ai/agent-framework/agent-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/agent-tool)).

The supervisor agent is responsible for creating and routing tool calls to each of your subagents, passing only the context necessary. You can modify this behavior and pass along the entire message history if desired. See the [LangGraph docs](https://langchain-ai.github.io/langgraph/reference/supervisor/) for more information.

### Write agent code to file

Define the agent code in a single cell below. This lets you write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.


In [0]:
%%writefile agent.py
import json
from typing import Generator, Literal
from uuid import uuid4

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from databricks_langchain.genie import GenieAgent
from langchain_core.runnables import Runnable
from langchain.agents import create_agent
from langgraph.graph.state import CompiledStateGraph
from langgraph_supervisor import create_supervisor
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from pydantic import BaseModel

client = DatabricksFunctionClient()
set_uc_function_client(client)

########################################
# Create your LangGraph Supervisor Agent
########################################

GENIE = "genie"


class ServedSubAgent(BaseModel):
    endpoint_name: str
    name: str
    task: Literal["agent/v1/responses", "agent/v1/chat", "agent/v2/chat"]
    description: str


class Genie(BaseModel):
    space_id: str
    name: str
    task: str = GENIE
    description: str


class InCodeSubAgent(BaseModel):
    tools: list[str]
    name: str
    description: str


TOOLS = []


def stringify_content(state):
    msgs = state["messages"]
    if isinstance(msgs[-1].content, list):
        msgs[-1].content = json.dumps(msgs[-1].content, indent=4)
    return {"messages": msgs}


def create_langgraph_supervisor(
    llm: Runnable,
    externally_served_agents: list[ServedSubAgent] = [],
    in_code_agents: list[InCodeSubAgent] = [],
):
    agents = []
    agent_descriptions = ""

    # Process inline code agents
    for agent in in_code_agents:
        agent_descriptions += f"- {agent.name}: {agent.description}\n"
        uc_toolkit = UCFunctionToolkit(function_names=agent.tools)
        TOOLS.extend(uc_toolkit.tools)
        agents.append(create_agent(llm, tools=uc_toolkit.tools, name=agent.name))

    # Process served endpoints and Genie Spaces
    for agent in externally_served_agents:
        agent_descriptions += f"- {agent.name}: {agent.description}\n"
        if isinstance(agent, Genie):
            # to better control the messages sent to the genie agent, you can use the `message_processor` param: https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_langchain.html#databricks_langchain.GenieAgent
            genie_agent = GenieAgent(
                genie_space_id=agent.space_id,
                genie_agent_name=agent.name,
                description=agent.description,
            )
            genie_agent.name = agent.name
            agents.append(genie_agent)
        else:
            model = ChatDatabricks(
                endpoint=agent.endpoint_name, use_responses_api="responses" in agent.task
            )
            # Disable streaming for subagents for ease of parsing
            model._stream = lambda x: model._stream(**x, stream=False)
            agents.append(
                create_agent(
                    model,
                    tools=[],
                    name=agent.name,
                    post_model_hook=stringify_content,
                )
            )

    # TODO: The supervisor prompt includes agent names/descriptions as well as general
    # instructions. You can modify this to improve quality or provide custom instructions.
    prompt = f"""
    You are a supervisor in a multi-agent system.

    1. Understand the user's last request
    2. Read through the entire chat history.
    3. If the answer to the user's last request is present in chat history, answer using information in the history.
    4. If the answer is not in the history, from the below list of agents, determine which agents are best suited to answer the question.
    5, Follow the agent route of “plan → choose tools → execute → reflect → respond.” 
    6, Show the thinking and planning process.
    5. Provide a summarized response to the user's last query, even if it's been answered before.

    {agent_descriptions}"""

    return create_supervisor(
        agents=agents,
        model=llm,
        prompt=prompt,
        add_handoff_messages=False,
        output_mode="full_history",
    ).compile()


##########################################
# Wrap LangGraph Supervisor as a ResponsesAgent
##########################################


class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        first_message = True
        seen_ids = set()

        # can adjust `recursion_limit` to limit looping: https://docs.langchain.com/oss/python/langgraph/GRAPH_RECURSION_LIMIT#troubleshooting
        for _, events in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates"]):
            new_msgs = [
                msg
                for v in events.values()
                for msg in v.get("messages", [])
                if msg.id not in seen_ids
            ]
            if first_message:
                seen_ids.update(msg.id for msg in new_msgs[: len(cc_msgs)])
                new_msgs = new_msgs[len(cc_msgs) :]
                first_message = False
            else:
                seen_ids.update(msg.id for msg in new_msgs)
                node_name = tuple(events.keys())[0]  # assumes one name per node
                yield ResponsesAgentStreamEvent(
                    type="response.output_item.done",
                    item=self.create_text_output_item(
                        text=f"<name>{node_name}</name>", id=str(uuid4())
                    ),
                )
            if len(new_msgs) > 0:
                yield from output_to_responses_items_stream(new_msgs)


#######################################################
# Configure the Foundation Model and Serving Sub-Agents
#######################################################

# TODO: Replace with your model serving endpoint

# 1. Super Agent, which will finally generate the ensemble code
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)


# TODO: Add the necessary information about each of your subagents. Subagents could be agents deployed to Model Serving endpoints or Genie Space subagents.
# Your agent descriptions are crucial for improving quality. Include as much detail as possible.
EXTERNALLY_SERVED_AGENTS = [
    Genie(
        space_id="01f0956a54af123e9cd23907e8167df9",
        name="Provider Enrollment",
        description="This agent can answer questions about provider and patient enrollment. \
            This dataset contains two tables: provider and enrollment. The provider table includes \
            information about healthcare claims, such as claim ID, patient ID, provider NPI, provider role, \
            and taxonomy code. \
            The enrollment table contains patient demographic and enrollment details, including gender, year of birth, ZIP code, state, enrollment dates, benefit type, and pay type.",
    ),
    Genie(
        space_id="01f0956a387714969edde65458dcc22a",
        name="Claims",
        description=(
            "This agent can answer questions about Medical and pharmacy claims. There are two "
            "tables: medical_claim and pharmacy_claim, both in the hv_claims_sample schema. Each "
            "table contains claims data with columns for claim_id, patient_id, date_service, and "
            "pay_type, among others. They can be connected by the patient_id column, which "
            "identifies the patient associated with each claim."
        ),
    ), 
    Genie(
        space_id="01f0956a4b0512e2a8aa325ffbac821b",
        name="Diagnosiss and Procedures",
        description=(
            "This agent can answer questions about diagnosiss and procedures. There are two tables: procedure and diagnosis, \
            both in the hv_claims_sample schema. They are connected by the columns claim_id and patient_id, which appear in \
            both tables and can be used to join procedure and diagnosis information for the same claim and patient."
        ),
    ),

    # ServedSubAgent(
    #     endpoint_name="cities-agent",
    #     name="city-agent", # choose a semantically relevant name for your agent
    #     task="agent/v1/responses",
    #     description="This agent can answer questions about the best cities to visit in the world.",
    # ),
]

############################################################
# Create additional agents in code
############################################################

# TODO: Fill the following with UC function-calling agents. The tools parameter is a list of UC function names that you want your agent to call.
IN_CODE_AGENTS = [
    InCodeSubAgent(
        tools=["system.ai.*"],
        name="code execution agent",
        description="The code execution agent specializes in solving programming challenges, generating code snippets, debugging issues, and explaining complex coding concepts.",
    )
]

#################################################
# Create supervisor and set up MLflow for tracing
#################################################

supervisor = create_langgraph_supervisor(llm, EXTERNALLY_SERVED_AGENTS, IN_CODE_AGENTS)

mlflow.langchain.autolog()
AGENT = LangGraphResponsesAgent(supervisor)
mlflow.models.set_model(AGENT)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-03c794ad-9a83-4013-983b-316bf38f5786/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


[Trace(trace_id=tr-29da8380cc9e2f2aae5ccfb4a4024879), Trace(trace_id=tr-bf24d0a1d63f590c92a4915fb39e281b)]

## Test the agent

Interact with the agent to test its output. Since this notebook called `mlflow.langchain.autolog()` you can view the trace for each step the agent takes.

Even if you didn't add any subagents in the agent definition above, the supervisor agent can still answer questions. It just won't have any subagents to switch to.

**Important:** LangGraph internally uses exceptions (something like `Command` or `ParentCommand`) to switch between nodes. These particular exceptions may appear in your MLflow traces as Events, but this behavior is expected and should not be a cause for concern.

In [0]:
 dbutils.library.restartPython()

In [0]:
 from agent import AGENT

# TODO: Replace this placeholder `input_example` with a domain-specific prompt for your agent.
input_example = {
    "input": [
        {"role": "user", "content": "what tools do you have access to"}
    ]
}


AGENT.predict(input_example)

ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='message', id='lc_run--512e294f-088a-47ce-b5b6-16327e4ad80f-0', content=[{'text': "I have access to three specialized agents that can help answer different types of questions:\n\n1. **Code Execution Agent** - This agent can help with programming challenges, generate code snippets, debug issues, and explain coding concepts.\n\n2. **GENIE_PATIENT** - This agent specializes in answering questions about patient information.\n\n3. **Medication** - This agent can provide information about medications.\n\nI can transfer your questions to the most appropriate agent based on what you're asking about. Is there a specific type of information you're looking for today?", 'type': 'output_text'}], role='assistant')], parallel_tool_calls=None, temperature=None, tools=None, top_p=None, max_output

Trace(trace_id=tr-9490a9956322a5dfaf4ac48f69d72cde)

In [0]:
for event in AGENT.predict_stream(input_example):
  print(event.model_dump(exclude_none=True))

{'type': 'response.output_item.done', 'item': {'id': 'lc_run--3be0df34-5f42-48e5-a7e6-6f705a786a22-0', 'content': [{'text': "I have access to three specialized agents that can help with different types of questions:\n\n1. **Code Execution Agent** - This agent can help with programming challenges, generate code snippets, debug issues, and explain coding concepts.\n\n2. **GENIE_PATIENT Agent** - This agent can answer questions about patient information and medical records.\n\n3. **Medication Agent** - This agent can provide information about medications, including details about dosages, side effects, interactions, and usage guidelines.\n\nI can transfer your questions to the most appropriate agent based on what you're asking about. Is there a specific topic you'd like assistance with?", 'type': 'output_text'}], 'role': 'assistant', 'type': 'message'}}


Trace(trace_id=tr-e0d1f2142ae2e96464dd94be3a259504)

Cross-room questions:
1. How many patients older than 50 years are on Voltaren?
2. 

In [0]:
# from agent import AGENT

# TODO: Replace this placeholder `input_example` with a domain-specific prompt for your agent.
input_example = {
    "input": [
        {"role": "user", "content": "How many patients older than 50 years are on Voltaren?"}
    ]
}


for event in AGENT.predict_stream(input_example):
  print(event.model_dump(exclude_none=True))

{'type': 'response.output_item.done', 'item': {'id': '7f359405-aeeb-479a-aca0-eb45ef492828', 'content': [{'text': '<name>Medication</name>', 'type': 'output_text'}], 'role': 'assistant', 'type': 'message'}}
{'type': 'response.output_item.done', 'item': {'id': '074994ce-81aa-42af-8877-69850177630c', 'content': [{'text': '|    |   patient_count |\n|---:|----------------:|\n|  0 |           65355 |', 'type': 'output_text'}], 'role': 'assistant', 'type': 'message'}}
{'type': 'response.output_item.done', 'item': {'id': '7623bb1a-8e0a-4504-84a3-1feb64d9d68d', 'content': [{'text': '<name>supervisor</name>', 'type': 'output_text'}], 'role': 'assistant', 'type': 'message'}}


Trace(trace_id=tr-b56f5f90d38f4c35de9dd920a4050a39)


Here’s what the Databricks LangChain GenieAgent returns and how to include SQL and control result format.

What GenieAgent returns
The GenieAgent function returns a LangChain RunnableLambda. When you invoke it, it returns a dict with a single key "messages" containing a list of AIMessage objects. 
1

By default, the list includes one AIMessage with the query output named "query_result"; if you set include_context=True, it also includes the reasoning ("query_reasoning") and the generated SQL ("query_sql"). 
1
2

Does it include message, SQL and result as CSV?
Messages: Yes—responses are packaged as AIMessage objects in a "messages" list as described above. 
1

SQL: Yes—but only if you pass include_context=True. In that case you’ll get an AIMessage named "query_sql" that contains the generated SQL string. 
1
2

Result format: By default the result is a string containing a Markdown table (not CSV). The underlying Genie implementation converts the result DataFrame to Markdown unless you request a pandas DataFrame via return_pandas=True on the databricks_ai_bridge.Genie class. 
3

CSV specifically: Not by default. If you need CSV, use the lower-level databricks_ai_bridge.Genie with return_pandas=True and then convert to CSV (e.g., df.to_csv(...)). Note: the LangChain GenieAgent helper doesn’t expose return_pandas; you’d instantiate and use the Genie class directly for that control. 
3

## Test GenieAgent Output Elements

In [0]:
from databricks_langchain.genie import GenieAgent
from langchain_core.messages import AIMessage

# Create the Genie agent and include reasoning + SQL in the response
agent = GenieAgent(
    genie_space_id="01f08f4d1f5f172ea825ec8c9a3c6064",
    genie_agent_name="Medication",
    include_context=True,
)
# Invoke the agent with a single user message (you can pass a full chat history too)
resp = agent.invoke({
    "messages": [
        {"role": "user", "content": "How many patients older than 50 years are on Voltaren?"}
    ]
})

# Extract "thinking" (reasoning), SQL, and final answer/result
thinking = None
sql = None
answer = None

for msg in resp["messages"]:
    # Each item is an AIMessage with a 'name' and 'content'
    if isinstance(msg, AIMessage):
        if msg.name == "query_reasoning":
            thinking = msg.content
        elif msg.name == "query_sql":
            sql = msg.content
        elif msg.name == "query_result":
            answer = msg.content

print("THINKING:\n", thinking)
print("\nSQL:\n", sql)
print("\nANSWER:\n", answer)

THINKING:
 You're looking for the number of distinct patients older than 50 years who have been prescribed Voltaren.

SQL:
 SELECT CASE WHEN COUNT(DISTINCT `PATIENT_ID`) < 10 THEN 'Count is less than 10' ELSE CAST(COUNT(DISTINCT `PATIENT_ID`) AS STRING) END AS patient_count
FROM `genie`.`dbo`.`medications`
WHERE `MEDICATION_NAME` ILIKE '%voltaren%'

ANSWER:
 |    |   patient_count |
|---:|----------------:|
|  0 |           65445 |


[Trace(trace_id=tr-c0b6275aaa372d2c87e6d9faadf07390), Trace(trace_id=tr-302577c38eab76e8d052d74dd251d556)]

## Test fetch Genie Metadata

In [0]:
import os
# TODO
# 1. replace with SP credentials, for dev just use from .env locally
os.environ["DATABRICKS_HOST"]
os.environ["DATABRICKS_TOKEN"]

In [0]:
#!/usr/bin/env python3
import os
import re
import json
import pathlib
import requests

HOST = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
TOKEN = os.environ.get("DATABRICKS_TOKEN") or os.environ.get("DATABRICKS_PAT")
if not HOST or not TOKEN:
    raise SystemExit("Please export DATABRICKS_HOST and DATABRICKS_TOKEN")

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}

In [0]:
%sql
create catalog if not exists `yyang`;
use catalog `yyang`;
create schema if not exists `multi_agent_genie`;
use schema `multi_agent_genie`;

create volume if not exists `yyang`.`multi_agent_genie`.`volume`;

In [0]:
OUTDIR = pathlib.Path("/Volumes/yyang/multi_agent_genie/volume/genie_exports")
print(OUTDIR)

/Volumes/databricks_c3od_poc/multi_agent_genie/volume/genie_exports


In [0]:
OUTDIR.mkdir(exist_ok=True, parents=True)

In [0]:
OUTDIR.__class__

pathlib.PosixPath

In [0]:
def safe_name(s: str) -> str:
    s = s.strip() or "untitled"
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)[:150]

def list_spaces():
    spaces = []
    params = {}
    while True:
        resp = requests.get(f"{HOST}/api/2.0/genie/spaces", headers=HEADERS, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        print(data)
        spaces.extend(data.get("spaces", []))
        token = data.get("next_page_token") or data.get("page_token") or None
        if not token:
            break
        params = {"page_token": token}
    return spaces

def export_space(space_id: str, title_hint: str = ""):
    url = f"{HOST}/api/2.0/genie/spaces/{space_id}"
    resp = requests.get(url, headers=HEADERS, params={"include_serialized_space": "true"}, timeout=120)
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        if resp.status_code == 403:
            print(f"403 Forbidden: {resp.url}")
            return
        raise
    obj = resp.json()

    title = obj.get("title") or title_hint or space_id
    base = f"{space_id}__{safe_name(title)}"
    raw_path = OUTDIR / f"{base}.space.json"
    raw_path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

    ser = obj.get("serialized_space")
    if ser:
        try:
            ser_obj = json.loads(ser)
            (OUTDIR / f"{base}.serialized.json").write_text(json.dumps(ser_obj, indent=2), encoding="utf-8")
        except json.JSONDecodeError:
            # Fallback: save raw string if not valid JSON
            (OUTDIR / f"{base}.serialized.txt").write_text(ser, encoding="utf-8")

    return base

def main():
    spaces = list_spaces()
    print(f"Found {len(spaces)} spaces")
    for sp in spaces:
        sid = sp.get("id") or sp.get("space_id")
        title = sp.get("title") or ""
        if not sid:
            continue
        base = export_space(sid, title)
        print(f"Exported {sid} -> {base}.space.json (+ serialized)")

if __name__ == "__main__":
    main()

{'spaces': [{'space_id': '01f08f4d1f5f172ea825ec8c9a3c6064', 'title': 'MEDICATIONS', 'description': 'These data describe the patients ordered (prescribed) medications, medication class, dates ordered and discontinued, medication class, is the medication is current or historical, route, strength and dosage', 'warehouse_id': 'cca0b29ed7524730'}, {'space_id': '01f08a9fd9ca125a986d01c1a7a5b2fe', 'title': 'GENIE_LABORATORY_BIOMARKERS', 'description': 'This space provided data on patient laboratory testing as well as genomic testing for cancer biomarkers.   It gives testing dates, results and other contextual information', 'warehouse_id': 'cca0b29ed7524730'}, {'space_id': '01f07795f6981dc4a99d62c9fc7c2caa', 'title': 'GENIE_TREATMENT', 'description': ' The space provided data on patient treatments.   This includes invasive treatment like surgeries and diagnostic procedures like radiological testing.   It also has data on cancer treatment plans as well as bone marrow transplants i.e. stem cell

In [0]:
import json
from langchain_core.documents import Document

json_path = "/Volumes/yyang/multi_agent_genie/volume/genie_exports/01f072dbd668159d99934dfd3b17f544__GENIE_PATIENT.space.json"

with open(json_path, "r", encoding="utf-8") as f:
    genie_json = json.load(f)

doc = Document(page_content=json.dumps(genie_json, indent=2), metadata={"source": json_path})
display(doc)

Document(metadata={'source': '/Volumes/databricks_c3od_poc/multi_agent_genie/volume/genie_exports/01f072dbd668159d99934dfd3b17f544__GENIE_PATIENT.space.json'}, page_content='{\n  "space_id": "01f072dbd668159d99934dfd3b17f544",\n  "title": "GENIE_PATIENT",\n  "description": "The space describes the cancer patient population, demographics, ECOG score, appointments and insurance data",\n  "warehouse_id": "cca0b29ed7524730",\n  "serialized_space": "{\\n  \\"version\\": 1,\\n  \\"config\\": {\\n    \\"sample_questions\\": [\\n      {\\n        \\"id\\": \\"01f0c17420951989882692eb2e4601f3\\",\\n        \\"question\\": [\\n          \\"How many patients are age 45 or greater?\\"\\n        ]\\n      },\\n      {\\n        \\"id\\": \\"01f073cfee761d96bd694d03e7bcd49a\\",\\n        \\"question\\": [\\n          \\"How many patients live in Johnson county?\\"\\n        ]\\n      },\\n      {\\n        \\"id\\": \\"01f073cfee761587955a2cb1b0faefbf\\",\\n        \\"question\\": [\\n          \\"H

In [0]:
import json
from langchain_core.documents import Document

json_path = "/Volumes/yyang/multi_agent_genie/volume/genie_exports/01f072dbd668159d99934dfd3b17f544__GENIE_PATIENT.space.json"

with open(json_path, "r", encoding="utf-8") as f:
    genie_json = json.load(f)

doc = Document(page_content=json.dumps(genie_json, indent=2), metadata={"source": json_path})
display(doc)



Based on the above parsed docs for a genie space, build vector search index for the space. Then Super Agent can use the vector search index to search the right space for the question.

Thinking and planning examples:

1. 
How many colon cancer patients were diagnosed between 2020 and 2024 who are currently taking any medication?

Analysis
You're looking for the count of unique colon cancer patients diagnosed between 2020 and 2024 who are currently taking any medication.
Understanding the question
currently taking any medication — The query assumes that 'currently taking any medication' refers to patients with an active diagnosis status related to medication. However, it could also mean patients taking specific medications or those with any medication-related diagnosis, which might yield different results.
Found relevant data
cancer_diagnosis
cancer_comorbidities
Calculated an answer based on these steps
Count the distinct patients diagnosed with colon cancer.
Filter for diagnoses made between 2020 and 2024.
Include only those patients who are currently taking medication.

2. 
